# CreditLens — Exploratory Data Analysis

Análisis exploratorio sobre el dataset **Give Me Some Credit** de Kaggle (150,000 registros reales de solicitudes de crédito). El objetivo es entender la estructura del problema antes de modelar, identificar problemas en los datos y justificar las decisiones de feature engineering que se implementaron en `dbt_project/models/mart/mart_credit_features.sql`.

**Variable objetivo:** `SeriousDlqin2yrs` — indica si el cliente entró en mora grave (90+ días) en los 2 años siguientes.

**Autor:** Juan Alvarez · Universidad de San Buenaventura
**Repositorio:** [github.com/JuanAlvarezgh/creditlens](https://github.com/JuanAlvarezgh/creditlens)

---

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 100
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 10

## 1. Carga e inspección inicial

In [ ]:
df = pd.read_csv("../data/cs-training.csv", index_col=0)
print(f"Shape: {df.shape[0]:,} filas × {df.shape[1]} columnas")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().T.round(2)

**Primeras observaciones:**

- **150,000 registros, 11 columnas** (10 features + 1 target).
- Tipos: `int64` para counts y target, `float64` para ratios e ingreso.
- Dos columnas con nulos: `MonthlyIncome` y `NumberOfDependents`.
- Valores extremos sospechosos a revisar:
  - `RevolvingUtilizationOfUnsecuredLines` máx ≈ 50,708 (debería estar entre 0 y 1)
  - `DebtRatio` máx ≈ 329,664 (debería ser un valor razonable)
  - `age` mín = 0 (imposible)
  - Las columnas de moras tienen máx = 98 (probablemente un código especial de Kaggle/origen)

Estos valores extremos se limpiarán en la capa `staging` de dbt antes de llegar al modelo.

## 2. Variable objetivo — desbalance de clases

In [ ]:
target_counts = df["SeriousDlqin2yrs"].value_counts()
default_rate = df["SeriousDlqin2yrs"].mean()

print(f"No Default (0): {target_counts[0]:>7,} ({(1-default_rate):.2%})")
print(f"Default    (1): {target_counts[1]:>7,} ({default_rate:.2%})")

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(["No Default", "Default"], target_counts.values,
              color=["#4C72B0", "#DD8452"], edgecolor="white", linewidth=1.5)
ax.set_title("Distribución de la variable objetivo")
ax.set_ylabel("Cantidad de clientes")
for bar, count in zip(bars, target_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1500,
            f"{count:,}", ha="center", fontsize=10)
plt.tight_layout()
plt.show()

**Desbalance significativo:** sólo el **~6.7%** de la población defaulteó.

**Implicaciones para el modelado:**

- No usar `accuracy` como métrica: un modelo trivial que prediga siempre "no default" tendría ~93% de accuracy y sería inútil.
- Métricas apropiadas: **AUC-ROC**, **KS Statistic**, **Gini** — el estándar bancario.
- Considerar técnicas de balance: `class_weight='balanced'`, SMOTE oversampling, o `scale_pos_weight` en XGBoost.
- El umbral de decisión por defecto (0.5) sesga hacia la clase mayoritaria; en producción el umbral se ajusta según el costo del banco de un falso negativo (default no detectado) vs falso positivo (cliente sano rechazado).

## 3. Valores faltantes

In [ ]:
nulls = df.isna().sum()
nulls_pct = (nulls / len(df) * 100).round(2)
nulls_df = pd.DataFrame({"Nulos": nulls, "Porcentaje": nulls_pct})
nulls_df = nulls_df[nulls_df["Nulos"] > 0].sort_values("Nulos", ascending=False)
nulls_df

**Análisis de nulos:**

- **`MonthlyIncome` (~19.8% nulo)** — porcentaje alto. Posibles causas en el mundo real: trabajadores informales que no reportan ingreso, datos no verificados, errores de captura. En este MVP los descartamos. En producción exploraríamos si la *ausencia* es predictiva por sí misma (los clientes sin ingreso reportado podrían tener más riesgo).
- **`NumberOfDependents` (~2.6% nulo)** — porcentaje bajo. Asumimos que la ausencia significa 0 dependientes y lo imputamos con 0 en la capa `staging` de dbt.

**Decisión:** drop rows con `MonthlyIncome` nulo en el productor Kafka (`dropna()`), imputar 0 para Dependents en dbt.

**Trade-off:** perdemos ~20% de los datos. Para producción real evaluaríamos imputación con modelo (KNN, MICE) o mediana por segmento socioeconómico.

## 4. Distribuciones univariadas

In [ ]:
# Cleaned subset for visualization (caps extreme outliers for readability)
clean = df.dropna(subset=["MonthlyIncome"]).copy()
clean["NumberOfDependents"] = clean["NumberOfDependents"].fillna(0)

clean_viz = clean[
    (clean["age"].between(18, 100))
    & (clean["RevolvingUtilizationOfUnsecuredLines"] <= 2)
    & (clean["DebtRatio"] <= 5)
    & (clean["MonthlyIncome"] <= clean["MonthlyIncome"].quantile(0.99))
]

features_to_plot = [
    ("age", "Edad"),
    ("MonthlyIncome", "Ingreso mensual (USD)"),
    ("DebtRatio", "Ratio de deuda (DTI)"),
    ("RevolvingUtilizationOfUnsecuredLines", "Utilización rotativa"),
    ("NumberOfOpenCreditLinesAndLoans", "Líneas de crédito abiertas"),
    ("NumberRealEstateLoansOrLines", "Préstamos hipotecarios"),
    ("NumberOfDependents", "Dependientes"),
    ("NumberOfTime30-59DaysPastDueNotWorse", "Moras 30-59 días"),
    ("NumberOfTimes90DaysLate", "Moras 90+ días"),
]

fig, axes = plt.subplots(3, 3, figsize=(14, 11))
for ax, (col, title) in zip(axes.flat, features_to_plot):
    ax.hist(clean_viz[col], bins=40, color="#4C72B0", edgecolor="white", linewidth=0.4)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("")
    ax.set_ylabel("")
plt.tight_layout()
plt.show()

**Observaciones clave:**

- **`age`** — distribución aproximadamente normal centrada en 50-55 años.
- **`MonthlyIncome`** — fuertemente sesgada a la derecha (típico en ingresos). Median bajo, cola larga de altos ingresos.
- **`DebtRatio`** — concentrado entre 0-1 (lo esperado), pero con cola larga de extremos.
- **`RevolvingUtilizationOfUnsecuredLines`** — distribución uniforme entre 0 y 1 como debería (es un porcentaje).
- **Moras (30-59, 90+)** — fuertemente sesgadas a 0: la mayoría tiene 0 moras, pocos tienen 1-2.
- **Líneas de crédito y hipotecas** — distribuciones de Poisson típicas para counts.

**Insight de modelado:** las features con sesgo fuerte (ingresos, ratios) podrían beneficiarse de transformaciones logarítmicas en una iteración futura. Los modelos de árboles (XGBoost, LightGBM) lo manejan internamente, así que en este MVP no transformamos.

## 5. Relación de cada feature con el target

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

features_bivariate = [
    ("age", "Edad"),
    ("MonthlyIncome", "Ingreso mensual"),
    ("DebtRatio", "Ratio de deuda"),
    ("RevolvingUtilizationOfUnsecuredLines", "Utilización rotativa"),
    ("NumberOfTimes90DaysLate", "Moras 90+ días"),
    ("NumberOfDependents", "Dependientes"),
]

for ax, (col, title) in zip(axes.flat, features_bivariate):
    for label, color, name in [(0, "#4C72B0", "No Default"), (1, "#DD8452", "Default")]:
        subset = clean_viz[clean_viz["SeriousDlqin2yrs"] == label][col]
        ax.hist(subset, bins=30, alpha=0.6, label=name, color=color, density=True)
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.set_ylabel("Densidad")
plt.tight_layout()
plt.show()

**Insights:**

- **`age`** — los clientes que defaultearon tienden a ser más jóvenes (curva naranja corrida a la izquierda).
- **`RevolvingUtilizationOfUnsecuredLines`** — diferencia muy marcada. Los defaulters tienen distribución concentrada cerca de 1.0 (uso del 100% del cupo), mientras los no-defaulters tienen utilizaciones más bajas. **Esta es probablemente la feature más predictiva.**
- **`NumberOfTimes90DaysLate`** — los defaulters tienen historial de moras 90+ días con mayor frecuencia. Poder predictivo alto.
- **`MonthlyIncome`** — diferencia sutil. Los defaulters tienden a tener ingresos un poco menores.
- **`DebtRatio`** y **`NumberOfDependents`** — diferencias visibles pero menos pronunciadas.

Esto se confirmará al ver el SHAP feature importance del modelo entrenado.

## 6. Tasa de default por bins

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# Age bins
df_age = clean[clean["age"].between(18, 90)].copy()
df_age["age_bin"] = pd.cut(df_age["age"], bins=[18, 25, 35, 45, 55, 65, 90])
age_default = df_age.groupby("age_bin")["SeriousDlqin2yrs"].mean()
age_default.plot(kind="bar", ax=axes[0, 0], color="#DD8452", edgecolor="white")
axes[0, 0].set_title("Tasa de default por edad")
axes[0, 0].set_ylabel("Default rate")
axes[0, 0].set_xlabel("Edad")
axes[0, 0].tick_params(axis="x", rotation=30)

# Income deciles
income_clean = clean[clean["MonthlyIncome"] < clean["MonthlyIncome"].quantile(0.99)].copy()
income_clean["income_decile"] = pd.qcut(income_clean["MonthlyIncome"], q=10,
                                         labels=[f"D{i}" for i in range(1, 11)])
income_default = income_clean.groupby("income_decile")["SeriousDlqin2yrs"].mean()
income_default.plot(kind="bar", ax=axes[0, 1], color="#DD8452", edgecolor="white")
axes[0, 1].set_title("Tasa de default por decil de ingreso (D1=bajo, D10=alto)")
axes[0, 1].set_ylabel("Default rate")
axes[0, 1].set_xlabel("Decil")

# Utilization bins
util_clean = clean[clean["RevolvingUtilizationOfUnsecuredLines"] <= 1.5].copy()
util_clean["util_bin"] = pd.cut(util_clean["RevolvingUtilizationOfUnsecuredLines"],
                                 bins=[-0.01, 0.1, 0.3, 0.5, 0.7, 0.9, 1.5],
                                 labels=["0-10%", "10-30%", "30-50%", "50-70%", "70-90%", "90%+"])
util_default = util_clean.groupby("util_bin")["SeriousDlqin2yrs"].mean()
util_default.plot(kind="bar", ax=axes[1, 0], color="#DD8452", edgecolor="white")
axes[1, 0].set_title("Tasa de default por utilización de crédito rotativo")
axes[1, 0].set_ylabel("Default rate")
axes[1, 0].set_xlabel("Rango de utilización")
axes[1, 0].tick_params(axis="x", rotation=30)

# 90+ days late
df_late = clean.copy()
df_late["late90_capped"] = df_late["NumberOfTimes90DaysLate"].clip(0, 5)
late_default = df_late.groupby("late90_capped")["SeriousDlqin2yrs"].mean()
late_default.plot(kind="bar", ax=axes[1, 1], color="#DD8452", edgecolor="white")
axes[1, 1].set_title("Tasa de default por # moras de 90+ días")
axes[1, 1].set_ylabel("Default rate")
axes[1, 1].set_xlabel("Moras 90+ días (truncado a 5)")

plt.tight_layout()
plt.show()

**Hallazgos cuantitativos:**

- **Edad:** los menores de 25 años tienen ~12% de default, vs ~3% en mayores de 65. **El riesgo se duplica/cuadruplica en clientes jóvenes.**
- **Ingreso:** los deciles más bajos (D1, D2) tienen tasas de default 2-3 veces mayores que los altos. Confirma la relación inversa esperada.
- **Utilización:** efecto dramático. **Clientes con utilización >90% defaultean a tasas de ~25-30%, vs ~3% en clientes con utilización <10%.** Esta es la variable con mayor poder discriminativo.
- **Moras 90+ días:** cada mora adicional incrementa significativamente la probabilidad de default. Quien ya tuvo ≥3 defaults previos tiene >50% de probabilidad de volver a defaultear.

Estos patrones validan el sentido común del dominio de credit risk y confirman que el problema tiene señal aprendible.

## 7. Matriz de correlación

In [ ]:
corr = clean.corr(method="spearman")  # Spearman captura relaciones monotónicas no lineales

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(
    corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
    square=True, linewidths=0.5, ax=ax,
    cbar_kws={"shrink": 0.8, "label": "Spearman ρ"},
    annot_kws={"size": 8},
)
ax.set_title("Matriz de correlación (Spearman)", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
target_corr = corr["SeriousDlqin2yrs"].drop("SeriousDlqin2yrs").sort_values(
    key=lambda s: s.abs(), ascending=False
)
print("Correlación con SeriousDlqin2yrs (Spearman, ordenadas por magnitud):\n")
for feat, c in target_corr.items():
    bar = "▍" * int(abs(c) * 50)
    print(f"  {feat:<45} {c:+.3f}  {bar}")

**Observaciones:**

- Las correlaciones no son extremadamente altas (max ≈ 0.28). Esto es típico en credit risk — el problema es genuinamente difícil y requiere modelos que capturen **interacciones** entre features, no solo efectos marginales.
- **Las moras 30-59, 60-89 y 90+ días están muy correlacionadas entre sí** (ρ ≈ 0.5-0.7). Esto motiva crear la feature derivada `total_late_payments` (suma) que captura el patrón general de incumplimiento, evitando colinealidad excesiva en modelos lineales.
- `NumberRealEstateLoansOrLines` y `NumberOfOpenCreditLinesAndLoans` también están correlacionadas — es esperable, ambos cuentan productos crediticios.
- La utilización rotativa, las moras 90+ y la edad son las top-3 predictoras univariadas. Lo confirmaremos con SHAP en el modelo entrenado.

## 8. Decisiones de feature engineering

Basado en el EDA anterior, estas son las features derivadas que implementamos en `dbt_project/models/mart/mart_credit_features.sql`:

### 8.1 `total_late_payments`

```sql
times_30_59_days_late + times_60_89_days_late + times_90_days_late AS total_late_payments
```

**Por qué:**
- Las tres columnas de moras tienen correlación inter-feature de 0.5-0.7. Suprimimos colinealidad para regresión logística.
- Una suma agregada captura el patrón general "este cliente tiene historial de incumplimiento" sin que importe la severidad específica.
- Los modelos de árboles pueden ignorar la suma si las features individuales son más útiles, así que no se pierde información.

### 8.2 `dti` (Debt-to-Income)

```sql
debt_ratio AS dti
```

**Por qué:**
- En este dataset `debt_ratio` ya está calculado como pagos de deuda / ingreso mensual (es la definición exacta de DTI).
- Lo renombramos para alinearlo con la terminología bancaria estándar (Bancolombia, Davivienda, FICO Score usan literalmente "DTI").
- Es la métrica regulatoria principal en credit scoring globalmente.

### 8.3 `utilization_segment` (categórico)

```sql
CASE
    WHEN revolving_utilization <= 0.3 THEN 'low'
    WHEN revolving_utilization <= 0.7 THEN 'medium'
    ELSE 'high'
END AS utilization_segment
```

**Por qué:**
- Los umbrales 30% y 70% son **estándar en la industria**: FICO recomienda <30% de utilización para no afectar el score; >70% se considera estrés financiero.
- Permite visualizaciones segmentadas claras en el dashboard (gráfico de pie del tab Resumen).
- En modelado avanzado permitiría hacer target encoding (Weight of Evidence) por segmento — técnica estándar bancaria.

### 8.4 Lo que NO hicimos y por qué

- **No log-transform de `MonthlyIncome`:** XGBoost/LightGBM manejan el sesgo internamente; solo sería necesario para regresión logística pura. Iteración futura.
- **No interactions (age × utilization, dti × dependents):** se evaluarán en Optuna search. Para MVP mantenemos features atómicas para preservar interpretabilidad.
- **No Weight of Evidence encoding:** próxima iteración. Cerraría la brecha entre Logistic Regression y los modelos de árboles.
- **No PSI (Population Stability Index) monitoring:** está en el roadmap para drift detection productivo.

## 9. Baseline rápido para set expectations

Antes de invertir tiempo en tuning, validemos que el problema tiene señal aprendible con un baseline mínimo.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

X = clean.drop(columns=["SeriousDlqin2yrs"]).fillna(0)
y = clean["SeriousDlqin2yrs"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

results = []
for name, model in [
    ("Logistic Regression", LogisticRegression(max_iter=1000)),
    ("Random Forest (100 trees)", RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
]:
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, proba)
    results.append({"Modelo": name, "AUC-ROC": round(auc, 4)})

pd.DataFrame(results)

**Baseline establecido:**

- **Logistic Regression** (~0.78 AUC) — modelo lineal capta la mayor parte de la señal global.
- **Random Forest** (~0.85 AUC) — modelos de árboles capturan interacciones no lineales que la regresión logística pierde, ganando ~7 puntos de AUC.

Esto confirma que la inversión en gradient boosting (XGBoost, LightGBM) está justificada para el pipeline productivo.

### Resultados finales del pipeline productivo

Después del entrenamiento completo en `ml/train.py` (5 modelos comparados sobre 118,624 registros de la capa mart):

| Algoritmo | AUC-ROC | KS Statistic | Gini |
|-----------|--------:|-------------:|-----:|
| **LightGBM** *(Production)* | **0.8563** | **0.5536** | **0.7126** |
| LightGBM (Tuned) | 0.8539 | 0.5534 | 0.7079 |
| XGBoost | 0.8536 | 0.5510 | 0.7071 |
| XGBoost (Deep) | 0.8503 | 0.5496 | 0.7006 |
| Logistic Regression | 0.7893 | 0.4546 | 0.5786 |

**KS de 0.55 y Gini de 0.71 son niveles que cualquier banco publicaría con orgullo en su Pilar 3 de Basilea.**

## 10. Conclusiones y próximos pasos

### Conclusiones del EDA

1. **Desbalance moderado** (6.7% default) — manejable con métricas correctas, sin necesidad obligatoria de SMOTE.
2. **Outliers significativos** en `RevolvingUtilization`, `DebtRatio` y `age` — limpiados en la capa `staging` de dbt.
3. **Datos faltantes en `MonthlyIncome`** — descartados; en producción evaluar imputación inteligente.
4. **La utilización de crédito rotativo es la feature más predictiva** — confirma la teoría de credit scoring.
5. **Las moras 30-59, 60-89, 90+ están correlacionadas** — agregadas en `total_late_payments`.
6. **El problema requiere modelos no lineales** — la regresión logística deja ~7 puntos de AUC en la mesa.

### Decisiones de modelado tomadas

- **Métricas:** AUC-ROC, KS Statistic, Gini (estándar regulatorio bancario).
- **Modelos comparados:** Logistic Regression (baseline), XGBoost (×2 configs), LightGBM (×2 configs).
- **Promoción automática** del mejor a Production via MLflow Model Registry.
- **Explicabilidad SHAP** en cada predicción (Top-3 features influyentes).

### Próximas iteraciones (roadmap)

- **Optuna search** para hyperparameter tuning (esperado +2-4 pts AUC).
- **Weight of Evidence encoding** para Logistic Regression (cierra la brecha con árboles).
- **Stacking ensemble** XGBoost + LightGBM + Logistic Regression.
- **PSI monitoring** para detección de drift en producción.
- **Calibración** con `CalibratedClassifierCV` para probabilidades regulatorias.

---

**Repositorio:** [github.com/JuanAlvarezgh/creditlens](https://github.com/JuanAlvarezgh/creditlens) ·
**LinkedIn:** [linkedin.com/in/juanalvarezgh](https://www.linkedin.com/in/juanalvarezgh)